# 🎙️ VoiceBatch Studio v2.0.6 - [Auto Sync & Language Lock]
यह वर्जन आपके GitHub से कनेक्ट होगा और फोल्डर अपने आप बनाएगा।

In [ ]:
# @title 📥 Step 1: GitHub टोकन और यूजरनेम यहाँ भरें
import os

GITHUB_USER = "Shriram nag" # @param {type:"string"}
GITHUB_TOKEN = "" # @param {type:"string"}
REPO_NAME = "VoiceBatch_Studio" # @param {type:"string"}

if GITHUB_TOKEN and GITHUB_USER:
    print("⏳ फोल्डर स्ट्रक्चर तैयार हो रहा है...")
    # फोल्डर अपने आप बनाना
    os.makedirs(f"{REPO_NAME}/configs", exist_ok=True)
    os.makedirs(f"{REPO_NAME}/models", exist_ok=True)
    
    !pip install -q gradio edge-tts librosa soundfile torchcodec coqui-tts
    print(f"✅ {REPO_NAME} फोल्डर बन गया और इंजन तैयार है!")
else:
    print("⚠️ भाई, पहले ऊपर टोकन और यूजरनेम भरें!")

In [ ]:
# @title 🚀 Step 2: app.py (Strict Language Fix)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def voice_pro_engine(text, audio_sample, speed, pitch, lang):
    if not audio_sample: return None
    
    # आउटपुट फोल्डर चेक करना
    out_dir = 'VoiceBatch_Studio/outputs'
    os.makedirs(out_dir, exist_ok=True)
    temp_out = f'{out_dir}/raw.wav'
    
    # 1000% Language Lock: यहाँ भाषा को कड़ाई से फिक्स किया गया है
    tts.tts_to_file(text=text, speaker_wav=audio_sample, language=lang, file_path=temp_out)
    
    y, sr = librosa.load(temp_out)
    y, _ = librosa.effects.trim(y, top_db=25)
    
    # Robotic Sound Fix
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    final_path = f'{out_dir}/v_batch_professional.wav'
    sf.write(final_path, y, sr)
    return final_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.0.6')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='स्क्रिप्ट लिखें', lines=5)
            smp = gr.Audio(label='वॉइस सैंपल', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr', 'bn', 'gu'], label='Language Lock', value='hi')
            spd = gr.Slider(0.85, 1.15, 1.0, step=0.01, label="Speed")
            ptc = gr.Slider(-2, 2, 0, step=1, label="Pitch")
            btn = gr.Button('Generate Realistic Voice 🚀', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Final Audio')

    btn.click(voice_pro_engine, [txt, smp, spd, ptc, lng], out)

demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप तैयार है। रन करें!")
!python app.py